# BNCI2014_001 Dataset Audit

Stage 1 audit for the approved BNCI2014_001 staged plan. This notebook checks MOABB access, dataset metadata, a small subject load, and the basic epoch contract before any modeling code is introduced.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import mne
import moabb
import numpy as np
import pandas as pd
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

moabb.set_log_level("ERROR")
pd.set_option("display.max_columns", 80)

ARTIFACT_DIR = Path("../artifacts/experiments/bnci2014_001").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_JSON = ARTIFACT_DIR / "stage1_dataset_audit.json"

print("moabb", getattr(moabb, "__version__", "unknown"))
print("mne", mne.__version__)
print("artifact", AUDIT_JSON)

moabb 1.5.0
mne 1.11.0
artifact /home/slauva/Projects/master-thesis-2024-2026/code/artifacts/experiments/bnci2014_001/stage1_dataset_audit.json


## Static MOABB Metadata

In [2]:
dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=4)

static_metadata = {
    "dataset_code": dataset.code,
    "subject_list": list(dataset.subject_list),
    "n_subjects": len(dataset.subject_list),
    "n_sessions": dataset.n_sessions,
    "event_id": dict(dataset.event_id),
    "interval": list(dataset.interval),
    "paradigm": dataset.paradigm,
}
static_metadata

{'dataset_code': 'BNCI2014-001',
 'subject_list': [1, 2, 3, 4, 5, 6, 7, 8, 9],
 'n_subjects': 9,
 'n_sessions': 2,
 'event_id': {'left_hand': 1, 'right_hand': 2, 'feet': 3, 'tongue': 4},
 'interval': [2, 6],
 'paradigm': 'imagery'}

## Small Subject Load

In [3]:
subjects_to_load = [1]
X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=subjects_to_load)
labels = np.asarray(labels)
metadata = metadata.reset_index(drop=True).assign(label=labels)

loaded_summary = {
    "subjects_loaded": subjects_to_load,
    "X_shape": list(X.shape),
    "X_dtype": str(X.dtype),
    "label_counts": dict(sorted(Counter(labels).items())),
    "metadata_columns": list(metadata.columns),
    "metadata_rows": int(len(metadata)),
}

display(pd.Series(loaded_summary, name="value"))
display(metadata.head())

subjects_loaded                                                   [1]
X_shape                                               [576, 22, 1001]
X_dtype                                                       float64
label_counts        {'feet': 144, 'left_hand': 144, 'right_hand': ...
metadata_columns                       [subject, session, run, label]
metadata_rows                                                     576
Name: value, dtype: object

,subject,session,run,label
0,1,0train,0,tongue
1,1,0train,0,feet
2,1,0train,0,right_hand
3,1,0train,0,left_hand
4,1,0train,0,left_hand


## Shape And Leakage-Relevant Metadata

In [4]:
if X.ndim != 3:
    raise AssertionError(f"Expected epochs as (trial, channel, time), got {X.shape}")
if len(labels) != X.shape[0] or len(metadata) != X.shape[0]:
    raise AssertionError("Epoch, label, and metadata row counts must match")

n_epochs, n_channels, n_times = X.shape
expected_subjects = {1}
observed_subjects = set(int(value) for value in metadata["subject"].unique())
if observed_subjects != expected_subjects:
    raise AssertionError(f"Expected subject subset {expected_subjects}, got {observed_subjects}")

epoch_summary = metadata.groupby(["subject", "session", "run", "label"], observed=True).size()
session_run_summary = metadata.groupby(["subject", "session", "run"], observed=True).size()

display(epoch_summary.rename("epochs").reset_index())
display(session_run_summary.rename("epochs").reset_index())

shape_contract = {
    "n_epochs": int(n_epochs),
    "n_channels": int(n_channels),
    "n_times": int(n_times),
    "label_set": sorted(str(value) for value in np.unique(labels)),
    "subjects": sorted(int(value) for value in metadata["subject"].unique()),
    "sessions": sorted(str(value) for value in metadata["session"].unique()),
    "runs": sorted(str(value) for value in metadata["run"].unique()),
}
shape_contract

,subject,session,run,label,epochs
0,1,0train,0,feet,12
1,1,0train,0,left_hand,12
2,1,0train,0,right_hand,12
3,1,0train,0,tongue,12
4,1,0train,1,feet,12
5,1,0train,1,left_hand,12
6,1,0train,1,right_hand,12
7,1,0train,1,tongue,12
8,1,0train,2,feet,12
9,1,0train,2,left_hand,12


,subject,session,run,epochs
0,1,0train,0,48
1,1,0train,1,48
2,1,0train,2,48
3,1,0train,3,48
4,1,0train,4,48
5,1,0train,5,48
6,1,1test,0,48
7,1,1test,1,48
8,1,1test,2,48
9,1,1test,3,48


{'n_epochs': 576,
 'n_channels': 22,
 'n_times': 1001,
 'label_set': ['feet', 'left_hand', 'right_hand', 'tongue'],
 'subjects': [1],
 'sessions': ['0train', '1test'],
 'runs': ['0', '1', '2', '3', '4', '5']}

## Raw-Level Channel And Sampling Check

In [5]:
raw_tree = dataset.get_data(subjects=subjects_to_load)
first_subject = subjects_to_load[0]
first_session = sorted(raw_tree[first_subject])[0]
first_run = sorted(raw_tree[first_subject][first_session])[0]
raw = raw_tree[first_subject][first_session][first_run]

raw_summary = {
    "raw_sfreq": float(raw.info["sfreq"]),
    "raw_n_channels": len(raw.ch_names),
    "raw_channel_types": dict(sorted(Counter(raw.get_channel_types()).items())),
    "first_channels": raw.ch_names[:10],
    "raw_duration_seconds": float(raw.times[-1]) if raw.n_times else 0.0,
}
raw_summary

{'raw_sfreq': 250.0,
 'raw_n_channels': 26,
 'raw_channel_types': {'eeg': 22, 'eog': 3, 'stim': 1},
 'first_channels': ['Fz',
  'FC3',
  'FC1',
  'FCz',
  'FC2',
  'FC4',
  'C5',
  'C3',
  'C1',
  'Cz'],
 'raw_duration_seconds': 386.936}

## Audit Artifact

In [6]:
audit = {
    "stage": "1 - MOABB Intake And Dataset Audit",
    "dataset": "BNCI2014_001",
    "static_metadata": static_metadata,
    "loaded_summary": loaded_summary,
    "shape_contract": shape_contract,
    "raw_summary": raw_summary,
    "notes": [
        "This stage performs no modeling.",
        "Subject identity, session, run, and epoch metadata are retained for later leakage audits.",
        "The primary planned protocol is subject-wise generalization.",
    ],
}
AUDIT_JSON.write_text(json.dumps(audit, indent=2, sort_keys=True), encoding="utf-8")
print(f"Wrote {AUDIT_JSON}")
print("BNCI2014_001_STAGE1_AUDIT_VERIFIED")

Wrote /home/slauva/Projects/master-thesis-2024-2026/code/artifacts/experiments/bnci2014_001/stage1_dataset_audit.json
BNCI2014_001_STAGE1_AUDIT_VERIFIED
